# Hopfion Eigenfrequency Analysis — Uniform Field
Processes Mumax3 micromagnetic simulation output, computes FMR spectra via FFT, detects eigenfrequency peaks, and visualises mode profiles across XY / XZ / YZ planes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from scipy.signal import find_peaks
from scipy import ndimage
from scipy.optimize import curve_fit
from IPython.display import clear_output

import mumax3PP.ovf as ovf
import mumax3PP.parameters as parameters
import mumax3PP.fft_across_xyzm as FFT_across_xyzm

## Configuration

In [ ]:
# Simulation parameters
comp   = "Bz"   # excitation field component: "Bx" or "Bz"
f_cut  = 20     # excitation frequency cut-off (GHz)
Ms     = 580    # saturation magnetisation (kA/m)
HOPFION_BOUNDARY = 420  # |m_x| contour level marking the hopfion boundary (kA/m)

# Paths
SIM_PATH  = f"/home/users/ksob/hopfions/eigenfreq/bloch_hopfion_r_100_nm_h_70_nm_{comp}_f_{f_cut}_GHz.out"
FIG_DIR   = "/home/users/ksob/hopfions/eigenfreq/figs"
FMR_DIR   = "/home/users/ksob/hopfions/eigenfreq"

## Helper functions

In [ ]:
def get_dt(times):
    """Estimate time step via linear fit (robust against irregular sampling)."""
    popt, _ = curve_fit(lambda x, a, b: a*x + b, range(len(times)), times,
                        p0=[np.abs(times[-1] - times[1]) / (len(times) - 1), 0])
    return popt[0]


def find_nearest(array, value):
    """Return (index, value) of the element in *array* closest to *value*."""
    idx = np.abs(array - value).argmin()
    return idx, array[idx]


def centred_axes(shape, step):
    """Return a zero-centred coordinate array for a given grid dimension and cell size."""
    ax = np.arange(shape) * step
    return ax - ax.mean()


def symmetric_norm(data):
    """TwoSlopeNorm centred on zero, scaled to ±max(|data|)."""
    vmax = np.abs(np.real(data)).max()
    return colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)


def imshow_mode(ax, data, extent_nm, norm=None, cmap='jet', mask=None):
    """Convenience wrapper for mode-profile imshow."""
    img = data * mask if mask is not None else data
    ax.imshow(img, aspect='equal', origin='lower', interpolation='gaussian',
              extent=extent_nm, cmap=cmap, norm=norm)


def add_hopfion_contour(ax, M0_slice, level, extent_nm, color='k'):
    """Overlay a hopfion boundary contour on *ax*."""
    ax.contour(M0_slice, levels=[level], colors=color, extent=extent_nm)

## 1 — Load equilibrium magnetisation

In [ ]:
parms  = parameters.ovfParms(head="min*")
M_file = ovf.OvfFile(SIM_PATH, parms)
print("Array shape (t,z,y,x,m):", M_file.array.shape)

M0  = M_file.array * Ms
cx  = M_file._headers["xstepsize"]

x = centred_axes(M0.shape[3], cx)
y = centred_axes(M0.shape[2], cx)
z = centred_axes(M0.shape[1], cx)

# Named extents for imshow (nm)
ext_xy = [x.min()*1e9, x.max()*1e9, y.min()*1e9, y.max()*1e9]
ext_xz = [x.min()*1e9, x.max()*1e9, z.min()*1e9, z.max()*1e9]
ext_yz = [y.min()*1e9, y.max()*1e9, z.min()*1e9, z.max()*1e9]

## 2 — Circular mask (disc geometry)

In [ ]:
# NaN outside the disc so the background is transparent
YY, XX = np.meshgrid(y, x, indexing='ij')
mask = np.where(np.sqrt(XX**2 + YY**2) <= x.max(), 1.0, np.nan)

## 3 — Equilibrium profile plots

In [ ]:
z0 = M0.shape[1] // 2   # mid-plane index

# XY mid-plane colour map of m_z
fig, ax = plt.subplots()
mz_slice = np.real(M0[0, z0, :, :, 2])
ax.imshow(mz_slice, aspect='equal', origin='lower', interpolation='gaussian',
          extent=ext_xy, cmap='jet', vmin=mz_slice.min(), vmax=mz_slice.max())
ax.contour(np.real(M0[0, z0, :, :, 0])**2 + np.real(M0[0, z0, :, :, 1])**2,
           levels=[HOPFION_BOUNDARY**2], colors='w', extent=ext_xy)
ax.set(xlabel='x (nm)', ylabel='y (nm)')
fig.savefig(f"{FIG_DIR}/new_contours.png", bbox_inches='tight', pad_inches=0)
plt.show()

# Cut-line profiles along x
fig, ax = plt.subplots()
for i, label in enumerate([r'$m_x$', r'$m_y$', r'$m_z$']):
    ax.plot(x * 1e9, np.real(M0[0, z0, y.size // 2, :, i]), label=label)
ax.set(xlabel=r'$\it{x}$ (nm)', ylabel=r'$m_i$ (kA/m)')
ax.legend()
fig.savefig(f"{FIG_DIR}/cutlines.png", bbox_inches='tight', pad_inches=0)
plt.show()

## 4 — Load dynamics and compute FFT

In [ ]:
import time

parms  = parameters.ovfParms(head="m00", tStart=1, tStop=500)
M_dyn  = ovf.OvfFile(SIM_PATH, parms)
M1, t1 = M_dyn.array, M_dyn.time
dt     = get_dt(t1)
print("Dynamic array shape:", M1.shape)

t0 = time.time()
MM, fqs = FFT_across_xyzm.FFT_across_xyzm(M1, t1)
print(f"FFT done in {(time.time()-t0)/60:.1f} min")

## 5 — Volume-averaged FMR spectrum

In [ ]:
# Spatial average → FFT magnitude
M_avg   = M1.mean(axis=(1, 2, 3))                        # shape (t, 3)
fft_amp = np.abs(np.fft.fft(M_avg, axis=0))             # shape (t, 3)
fff     = np.fft.fftfreq(fft_amp.shape[0], dt)

f0, f_i = 1, fff.shape[0] // 2                          # positive-frequency slice

# Select component matching the excitation
comp_idx = {'Bx': 0, 'Bz': 2}[comp]
fmr_i    = fft_amp[f0:f_i, comp_idx]
fmr_i   /= fmr_i.max()

# Peak detection
threshold = 0.35
peaks, _  = find_peaks(fmr_i, threshold)

fig, ax = plt.subplots()
ax.plot(fff[f0:f_i] * 1e-9, fmr_i)
ax.axhline(threshold, ls='--', c='k', lw=0.8)
for p in peaks:
    ax.axvline(fqs[p] * 1e-9, c='r', ls='--', lw=0.5)
ax.set(xlabel='Frequency (GHz)', ylabel='Normalised amplitude', ylim=(0, 1.05))
plt.show()

print("Peak indices:", peaks)
print("Peak frequencies (GHz):", np.round(fqs[peaks] * 1e-9, 2))

## 6 — Mode profile panels (Re part, all three planes)

In [ ]:
x0 = MM.shape[3] // 2
y0 = MM.shape[2] // 2
z0 = MM.shape[1] // 2

# Hopfion boundary slices (precomputed for speed)
bnd_xy = np.real(M0[0, z0, :, :, 0])**2 + np.real(M0[0, z0, :, :, 1])**2
bnd_xz = np.abs(M0[0, :, y0, :, 0])
bnd_yz = np.abs(M0[0, :, :, x0, 0])

COMP_LABELS = [r'$m_x$', r'$m_y$', r'$m_z$']

def plot_mode_panels(peak_idx, cmap='jet', use_abs=False, save=True):
    """3×4 panel figure: XY / XZ / YZ slices of all three m-components + spectrum."""
    freq_GHz = round(fqs[peak_idx] / 1e9, 2)
    mode     = MM[peak_idx]

    fig = plt.figure(figsize=(12, 7), constrained_layout=True)
    fig.suptitle(f"Uniform field {comp}, f = {freq_GHz} GHz", fontsize=14)
    spec = fig.add_gridspec(ncols=4, nrows=3)

    plane_cfg = [
        # (row, data_fn, extent,  contour_data,  contour_level, xlabel, ylabel, label)
        (0, lambda c: mode[z0, :, :, c],  ext_xy, bnd_xy, HOPFION_BOUNDARY**2, 'x (nm)', 'y (nm)', 'XY'),
        (1, lambda c: mode[:, y0, :, c],  ext_xz, bnd_xz, HOPFION_BOUNDARY,   'x (nm)', 'z (nm)', 'XZ'),
        (2, lambda c: mode[:, :, x0, c],  ext_yz, bnd_yz, HOPFION_BOUNDARY,   'y (nm)', 'z (nm)', 'YZ'),
    ]

    for row, data_fn, ext, cont_data, cont_lvl, xl, yl, plane_lbl in plane_cfg:
        for col in range(3):
            ax   = fig.add_subplot(spec[row, col])
            data = data_fn(col)
            img  = np.abs(np.real(data)) if use_abs else np.real(data)
            norm = None if use_abs else symmetric_norm(data)
            msk  = mask if row == 0 else None
            imshow_mode(ax, img, ext, norm=norm, cmap=cmap, mask=msk)
            add_hopfion_contour(ax, cont_data, cont_lvl, ext)
            ax.set(title=COMP_LABELS[col], xlabel=xl, ylabel=yl)
        fig.add_subplot(spec[row, 0]).text(
            ext[0] * 0.8, ext[3] * 0.85, f"{plane_lbl}-plane", fontsize=11)

    # Spectrum panel
    ax_s = fig.add_subplot(spec[:, 3])
    ax_s.plot(fff[f0:f_i] * 1e-9, fmr_i)
    ax_s.axvline(freq_GHz, lw=0.5, ls='--', color='r')
    ax_s.set(title=f'f = {freq_GHz} GHz', xlabel='frequency (GHz)', ylim=(0, 1.1))

    if save:
        fig.savefig(f"{FIG_DIR}/uni_field_{comp}_f_{freq_GHz}_GHz_fcut_{f_cut}_GHz.png",
                    bbox_inches='tight', pad_inches=0)
    plt.show()


for peak in peaks:
    plot_mode_panels(peak)

## 7 — Amplitude-only panels (|m|)

In [ ]:
for peak in peaks:
    plot_mode_panels(peak, cmap='inferno', use_abs=True, save=False)

## 8 — Phase animation of a selected mode

In [ ]:
def animate_mode(peak_idx, n_frames=100, plane='xy', save_dir=None):
    """Step through *n_frames* phase values and display (optionally save) each frame."""
    mode   = MM[peak_idx]
    phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

    for i, phi in enumerate(phases):
        mode_phi = mode * np.exp(1j * phi)

        if plane == 'xy':
            data, ext = np.real(mode_phi[z0, :, :, 2]) * mask, ext_xy
            xlabel, ylabel = 'x (nm)', 'y (nm)'
        else:  # yz
            data, ext = np.real(mode_phi[:, :, x0, 2]), ext_yz
            xlabel, ylabel = 'y (nm)', 'z (nm)'

        fig, ax = plt.subplots()
        ax.imshow(data, aspect='equal', origin='lower', interpolation='gaussian',
                  norm=symmetric_norm(data), extent=ext, cmap='jet')
        ax.set(xlabel=xlabel, ylabel=ylabel,
               title=fr'Phase: {phi/np.pi:.3f}$\pi$')

        if save_dir:
            fig.savefig(f"{save_dir}/{i:04d}.png", bbox_inches='tight', pad_inches=0)
        plt.show()
        clear_output(wait=True)


# Example: animate peak at index 669 in XY plane
# animate_mode(669, n_frames=100, plane='xy', save_dir="/home/users/ksob/hopfions/figures/modes")

## 9 — Radial localisation of modes

In [ ]:
YY, XX = np.meshgrid(y, x, indexing='ij')
R      = np.sqrt(XX**2 + YY**2)

def mode_radius(mode_mz_slice):
    """Amplitude-weighted mean radius of a 2-D mode slice."""
    amp = np.abs(mode_mz_slice)
    return (R * amp).sum() / amp.sum()

r_hop = np.array([mode_radius(MM[a, z0, :, :, 2]) for a in range(MM.shape[0])])

fig, ax = plt.subplots()
ax.scatter(fqs * 1e-9, r_hop * 1e9, s=4)
ax.set(xlabel='f (GHz)', ylabel='r (nm)', ylim=(35, 95))
plt.show()

## 10 — Compare Bx vs Bz FMR spectra from saved files

In [ ]:
dane = {}
for label in ('Bx', 'Bz'):
    d = np.loadtxt(f"{FMR_DIR}/fmr_{label}.txt", skiprows=1)
    dane[label] = d[:d.shape[0] // 2]   # positive-frequency half

f_axis = dane['Bx'][:, 0] * 1e-9
font   = 12
XLIM   = (0, f_axis.max())
XTICKS = np.arange(0, 20.1, 5)

comp_specs = [
    (1, 'x', dict(ylim=(0, 0.0085), yticks=np.arange(0, 0.0081, 0.002))),
    (2, 'y', dict(ylim=(0, 0.0085), yticks=np.arange(0, 0.0081, 0.002))),
    (3, 'z', dict(ylim=(0, 0.0021), yticks=np.arange(0, 0.0021, 0.0004))),
]

for col_idx, comp_lbl, lim_kw in comp_specs:
    fig, ax = plt.subplots()
    ax.set_title(f"FMR, {comp_lbl}-component", fontsize=font)
    for label in ('Bx', 'Bz'):
        ax.plot(f_axis, dane[label][:, col_idx], label=rf'$B_{label[1]}$')
    ax.legend(fontsize=font)
    ax.set_xlim(*XLIM)
    ax.set_xlabel(r'$\it{f}$ (GHz)', fontsize=font)
    ax.set_xticks(XTICKS)
    ax.set_ylim(0, lim_kw['ylim'][1])
    ax.set_yticks(lim_kw['yticks'])
    ax.tick_params(labelsize=font)
    fig.savefig(f"{FIG_DIR}/fig_fmr_{comp_lbl}_comp.pdf", pad_inches=0.01, bbox_inches='tight')
    plt.show()